# Analysis on combined Signal

In [ ]:
import pickle
import numpy as np
import os 
import sys
import json
import matplotlib.pyplot as plt

n_reps = 8
run_name = "emg_recording_20260206_145245_divided"
OUTPUT_FOLDER = fr"C:\Users\velar\SynologyDrive\Personal\ExperimentalData\0602026Session1\{run_name}\processed_{n_reps}reps"

CHANNEL_TO_REMOVE = [0,1,2,3]

In [ ]:
file= rf"C:\Users\velar\SynologyDrive\Personal\ExperimentalData\0602026Session1\{run_name}\move_005.pkl"

with open(file, 'rb') as f:
    data = pickle.load(f)
emg_data = data['data']
emg_data = emg_data
emg_data = np.delete(emg_data, CHANNEL_TO_REMOVE, axis=0)


plt.plot(emg_data[5,:])

In [ ]:
reps = np.arange(2,2+n_reps)
all_emg_data = None
all_emg_pause = None

for rep in reps:

    file= rf"C:\Users\velar\SynologyDrive\Personal\ExperimentalData\0602026Session1\{run_name}\move_{rep:03d}.pkl"

    with open(file, 'rb') as f:
        data = pickle.load(f)
    emg_data = data['data']
    temp = emg_data
    emg_data = emg_data[:,5000:12000]
    emg_data = np.delete(emg_data, CHANNEL_TO_REMOVE, axis=0)
    temp = np.delete(temp, CHANNEL_TO_REMOVE, axis=0)

    if all_emg_data is None:
        all_emg_data = emg_data
        all_emg_pause = temp
    else:
        all_emg_data = np.concatenate((all_emg_data, emg_data), axis=1)
        all_emg_pause = np.concatenate((all_emg_pause, temp), axis=1)

# Save the combined EMG data to a new file
check_folder = os.path.exists(OUTPUT_FOLDER)
if not check_folder:
    os.makedirs(OUTPUT_FOLDER)
    
output_file = os.path.join(OUTPUT_FOLDER, "concatenated_emg_channels.pkl")

with open(output_file, 'wb') as f:
    pickle.dump({'emg_data': all_emg_data, 'emg_pause': all_emg_pause}, f)
print(f"Combined EMG data saved to {output_file}")


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(all_emg_data[5,:])
plt.title("Combined EMG Data")
plt.xlabel("Time (samples)")
plt.ylabel("Amplitude")
plt.grid()

emg = all_emg_data.copy()

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

def remove_spikes(emg, threshold_std=5):
    """
    Remove large spikes via simple thresholding and linear interpolation.
    
    Parameters
    ----------
    emg : ndarray, shape (n_channels, n_samples)
    threshold_std : float
        Number of standard deviations for threshold
    
    Returns
    -------
    emg_clean : ndarray
    spike_mask : ndarray, bool
    """
    emg_clean = emg.copy()
    spike_mask = np.zeros_like(emg, dtype=bool)
    
    for ch in range(emg.shape[0]):
        signal = emg[ch]
        threshold = threshold_std * np.std(signal)
        
        spikes = np.abs(signal) > threshold
        spike_mask[ch] = spikes
        
        if np.any(spikes):
            clean_idx = np.where(~spikes)[0]
            spike_idx = np.where(spikes)[0]
            
            if len(clean_idx) > 1:
                interp = interp1d(clean_idx, signal[clean_idx], 
                                  kind='linear', bounds_error=False, 
                                  fill_value='extrapolate')
                emg_clean[ch, spike_idx] = interp(spike_idx)
    
    return emg_clean, spike_mask


In [ ]:
import pickle as pkl
import sys
import os

sys.path.append(os.path.abspath('..'))


# Custom modules
from muniverse.algorithms.decomposition import decompose_cbss


algo_cfg = r"C:\Users\velar\SynologyDrive\Personal\Thesis Code\src\configs\cbss.json"




emg_raw = emg.copy()
emg_clean, spikes = remove_spikes(emg_raw, threshold_std=8)

    

#plot emg before and after spike removal for first channel
# import matplotlib.pyplot as plt
# plt.figure(figsize=(12, 6))        
# plt.subplot(2, 1, 1)
# plt.plot(emg_raw[0], label='Raw EMG', color='red')
# plt.title(f'Raw EMG Signal - Channel 0')
# plt.xlabel('Samples')
# plt.ylabel('Amplitude')
# plt.legend()

# plt.subplot(2, 1, 2)
# plt.plot(emg_clean[0], label='Cleaned EMG', color='blue')
# plt.title(f'Cleaned EMG Signal - Channel 0 ')
# plt.xlabel('Samples')
# plt.ylabel('Amplitude')
# plt.legend()

# plt.tight_layout()
# plt.show()
    
    
results, metadata = decompose_cbss(
        data=emg_clean,
        algorithm_config=algo_cfg,
        show_config=False, 
)

#check if any in results is None
if any(value is None for value in results.values()):
    raise RuntimeError(f"Decomposition failed, skipping saving results ...")
    


print(f"Decomposed, saving results ...")


#check if output folder exists, if not create it    
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
trial_name = "combined_emg_data"

print(f"Saving results to {os.path.join(OUTPUT_FOLDER, f'{trial_name}.pkl')} result keys: {list(results.keys())}")

with open(os.path.join(OUTPUT_FOLDER, f"{trial_name}.pkl"), "wb") as f:
    pkl.dump((results, metadata), f)

In [ ]:
results["spikes"].keys()

In [ ]:
import matplotlib.pyplot as plt


def calculate_spike_triggered_average(emg_signal, spike_times, fs=2048, window_ms=40):
    """
    Calculate spike-triggered average waveform and per-channel SNR.

    Args:
        emg_signal: EMG signal array of shape (n_channels, n_timepoints)
        spike_times: Array of spike times in samples
        fs: Sampling frequency in Hz
        window_ms: Window size around spike in milliseconds (total window, centered on spike)

    Returns:
        sta: Spike-triggered average waveform (n_channels, window_samples)
        time_axis: Time axis in milliseconds relative to spike
        n_spikes_used: Number of spikes used (excludes those too close to edges)
        snr_db: Per-channel SNR in dB, computed as 10*log10(||STA||^2 / mean(||wf_i - STA||^2))
    """
    # Calculate window in samples
    window_samples = int((window_ms / 1000) * fs)
    half_window = window_samples // 2

    # Get dimensions
    n_channels, n_timepoints = emg_signal.shape

    # Initialize array to collect waveforms
    waveforms = []

    # Extract waveforms around each spike
    for spike in spike_times:
        spike_idx = int(spike)
        start_idx = spike_idx - half_window
        end_idx = spike_idx + half_window

        # Check if window is within signal bounds
        if start_idx >= 0 and end_idx < n_timepoints:
            waveform = emg_signal[:, start_idx:end_idx]
            waveforms.append(waveform)

    # Calculate average and SNR
    if len(waveforms) > 0:
        waveforms = np.array(waveforms)  # shape: (n_spikes, n_channels, window_samples)
        sta = np.mean(waveforms, axis=0)  # shape: (n_channels, window_samples)
        actual_samples = sta.shape[-1]

        # Residual noise SNR per channel
        # signal power = mean of STA squared (per channel)
        signal_power = np.mean(sta ** 2, axis=-1)  # (n_channels,)
        # noise power = mean residual variance across spikes (per channel)
        residuals = waveforms - sta[np.newaxis, :, :]  # (n_spikes, n_channels, window_samples)
        noise_power = np.mean(np.mean(residuals ** 2, axis=-1), axis=0)  # (n_channels,)
        # SNR in dB, guard against division by zero
        with np.errstate(divide='ignore', invalid='ignore'):
            snr_db = 10 * np.log10(signal_power / noise_power)
            snr_db = np.where(np.isfinite(snr_db), snr_db, 0.0)
    else:
        sta = np.zeros((n_channels, window_samples))
        actual_samples = window_samples
        snr_db = np.zeros(n_channels)

    # Create time axis in milliseconds, centered at 0
    time_axis = np.linspace(0, window_ms, actual_samples)

    return sta, time_axis, len(waveforms), snr_db


def plot_sta_grid(sta, time_axis, title="Spike-Triggered Average", grid_shape=None, snr_db=None):
    """
    Plot STA as a grid of electrode positions.

    Args:
        sta: STA waveform of shape (n_channels, window_samples)
        time_axis: Time axis in milliseconds
        title: Plot title
        grid_shape: Tuple (n_rows, n_cols) for arranging channels in a grid.
                   If None, arranges channels in a square-ish grid.
        snr_db: Optional per-channel SNR in dB. If provided, shown in subplot titles.
    """
    n_channels, window_samples = sta.shape

    # Determine grid layout
    if grid_shape is not None:
        n_rows, n_cols = grid_shape
        if n_rows * n_cols < n_channels:
            raise ValueError(f"Grid shape {grid_shape} too small for {n_channels} channels")
    else:
        # Arrange in roughly square grid
        n_cols = int(np.ceil(np.sqrt(n_channels)))
        n_rows = int(np.ceil(n_channels / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*1.5, n_rows*1.5))

    # Handle single row or column case
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)

    # Find global min/max for consistent y-axis
    vmin, vmax = sta.min(), sta.max()

    channel_idx = 0
    for row in range(n_rows):
        for col in range(n_cols):
            ax = axes[row, col]

            if channel_idx < n_channels:
                # Plot channel data
                ax.plot(time_axis, sta[channel_idx, :], 'b-', linewidth=0.8)
                ax.axvline(time_axis[len(time_axis)//2], color='r', linestyle='--', alpha=0.3, linewidth=0.5)
                ax.set_ylim(vmin, vmax)

                # Show SNR in title if available
                if snr_db is not None:
                    ax.set_title(f'Ch{channel_idx} ({snr_db[channel_idx]:.1f}dB)', fontsize=7, pad=2)
                else:
                    ax.set_title(f'Ch{channel_idx}', fontsize=8, pad=2)
                channel_idx += 1
            else:
                # Hide unused subplots
                ax.axis('off')

            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle(title, fontsize=14, y=0.995)
    plt.tight_layout()

    return fig

In [ ]:
fs = 2000
window_ms = 50
stas_dict = {}

spike_times = results["spikes"]
for n_idx, neuron in spike_times.items():

    sta, time_axis, n_spikes_used, snr_db = calculate_spike_triggered_average(
        emg_clean, neuron, fs=fs, window_ms=window_ms
    )

    time_axis = time_axis


    neuron_idx = n_idx

    stas_dict[neuron_idx] = {
        'sta': sta,
        'time_axis': time_axis,
        'n_spikes': len(neuron),
        'n_spikes_used': n_spikes_used,
        'snr_db': snr_db,
    }

    print(f"Neuron {n_idx}: {n_spikes_used} spikes, mean SNR = {np.mean(snr_db):.1f} dB")
    print(f"  Per-channel SNR (dB): {np.round(snr_db, 1)}")
    print(sta.shape)


    fig = plot_sta_grid(stas_dict[neuron_idx]['sta'], stas_dict[neuron_idx]['time_axis'],
                        title="MUAPs", snr_db=stas_dict[neuron_idx]['snr_db'])
    fig.savefig(os.path.join(OUTPUT_FOLDER, f"sta_neuron_{n_idx}.png"), dpi=300)
    plt.show()

filename = f"sta_{trial_name}.pkl"
with open(os.path.join(OUTPUT_FOLDER, filename), "wb") as f:
    pkl.dump(stas_dict, f)

# Analysis on Whole signal

In [ ]:
FILENAME = rf"C:\Users\velar\SynologyDrive\Personal\ExperimentalData\0602026Session1\{run_name}\move_001.pkl"


with open(FILENAME, 'rb') as f:
    data = pickle.load(f)
    
print(data["data"].shape)

#remove channels

emg = data["data"]
emg = np.delete(emg, CHANNEL_TO_REMOVE, axis=0)
emg = emg[:, 5000:] #keep only first 8 channels
plt.plot(emg[0])



In [ ]:
import pickle as pkl
import sys
import os

sys.path.append(os.path.abspath('..'))


# Custom modules
from muniverse.algorithms.decomposition import decompose_cbss


algo_cfg = r"C:\Users\velar\SynologyDrive\Personal\Thesis Code\src\configs\cbss.json"




emg_raw = emg.copy()
emg_clean, spikes = remove_spikes(emg_raw, threshold_std=8)

    

#plot emg before and after spike removal for first channel
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))        
plt.subplot(2, 1, 1)
plt.plot(emg_raw[0], label='Raw EMG', color='red')
plt.title(f'Raw EMG Signal - Channel 0')
plt.xlabel('Samples')
plt.ylabel('Amplitude')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(emg_clean[0], label='Cleaned EMG', color='blue')
plt.title(f'Cleaned EMG Signal - Channel 0 ')
plt.xlabel('Samples')
plt.ylabel('Amplitude')
plt.legend()

plt.tight_layout()
plt.show()
    
    
results_full, metadata_full = decompose_cbss(
        data=emg_clean,
        algorithm_config=algo_cfg,
        show_config=False, 
)

#check if any in results is None
if any(value is None for value in results_full.values()):
    raise RuntimeError(f"Decomposition failed, skipping saving results ...")
    


print(f"Decomposed, saving results ...")

#take only the last part of the filename after the last slash and remove the .pkl extension to get the trial name
trial_name = os.path.basename(FILENAME).replace('.pkl', '')
#check if output folder exists, if not create it    
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
trial_name = "full_emg_data"

print(f"Saving results to {os.path.join(OUTPUT_FOLDER, f'{trial_name}.pkl')} result keys: {list(results_full.keys())}")

with open(os.path.join(OUTPUT_FOLDER, f"{trial_name}.pkl"), "wb") as f:
    pkl.dump((results_full, metadata_full), f)

In [ ]:
print(results_full["spikes"].keys())

In [ ]:
fs = 2000
window_ms = 50
stas_dict_full = {}

spike_times = results_full["spikes"]
for n_idx, neuron in spike_times.items():

    sta, time_axis, n_spikes_used, snr_db = calculate_spike_triggered_average(
        emg_clean, neuron, fs=fs, window_ms=window_ms
    )

    #remove channels 2 and 16 (index 1 and 15)
    sta = np.delete(sta, [2, 16], axis=0)
    snr_db = np.delete(snr_db, [2, 16])
    time_axis = time_axis


    neuron_idx = n_idx

    stas_dict_full[neuron_idx] = {
        'sta': sta,
        'time_axis': time_axis,
        'n_spikes': len(neuron),
        'n_spikes_used': n_spikes_used,
        'snr_db': snr_db,
    }

    print(f"Neuron {n_idx}: {n_spikes_used} spikes, mean SNR = {np.mean(snr_db):.1f} dB")
    print(f"  Per-channel SNR (dB): {np.round(snr_db, 1)}")
    print(sta.shape)


    fig = plot_sta_grid(stas_dict_full[neuron_idx]['sta'], stas_dict_full[neuron_idx]['time_axis'],
                        title="MUAPs", snr_db=stas_dict_full[neuron_idx]['snr_db'])
    plt.savefig(os.path.join(OUTPUT_FOLDER, f"sta_neuron_all_{n_idx}.png"), dpi=300)
    plt.show()
    

filename = f"sta_{trial_name}.pkl"
with open(os.path.join(OUTPUT_FOLDER, filename), "wb") as f:
    pkl.dump(stas_dict_full, f)

# Tracking Analysis

In [ ]:
from scipy.signal import correlate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def norm_twod_xcorr(sta1, sta2, debug_plot=False, title=""):
    """
    Compute normalized 1D cross-correlation between two STAs using the channel
    with greatest amplitude and its immediate neighbors from each STA.
    """
    n_channels = sta1.shape[0]
    
    # Find channel with greatest amplitude (peak-to-peak) for each STA
    amp1 = np.ptp(sta1, axis=1)
    amp2 = np.ptp(sta2, axis=1)
    
    best_ch1 = np.argmax(amp1)
    best_ch2 = np.argmax(amp2)
    
    def get_channel_group(best_ch, n_ch):
        channels = [best_ch]
        if best_ch > 0:
            channels.insert(0, best_ch - 1)
        if best_ch < n_ch - 1:
            channels.append(best_ch + 1)
        return channels
    
    ch_group1 = get_channel_group(best_ch1, n_channels)
    ch_group2 = get_channel_group(best_ch2, sta2.shape[0])
    
    waveform1 = sta1[ch_group1, :].flatten()
    waveform2 = sta2[ch_group2, :].flatten()
    
    logger.debug(f"STA1 best channel: {best_ch1}, using channels {ch_group1}")
    logger.debug(f"STA2 best channel: {best_ch2}, using channels {ch_group2}")
    
    wf1_norm = (waveform1 - np.mean(waveform1)) / (np.std(waveform1) + 1e-10)
    wf2_norm = (waveform2 - np.mean(waveform2)) / (np.std(waveform2) + 1e-10)
    
    xcorr = correlate(wf1_norm, wf2_norm, mode='full')
    xcorr_normalized = xcorr / len(waveform1)
    
    max_xcorr = np.max(xcorr_normalized)
    max_lag = np.argmax(xcorr_normalized) - (len(waveform1) - 1)
    
    logger.debug(f"XCorr max: {max_xcorr:.4f} at lag {max_lag}")
    
    if debug_plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        for i, ch in enumerate(ch_group1):
            label = f'Ch {ch}' + (' (max)' if ch == best_ch1 else '')
            axes[0].plot(sta1[ch, :], linewidth=1.5 if ch == best_ch1 else 1, 
                        alpha=1.0 if ch == best_ch1 else 0.6, label=label)
        axes[0].set_title(f'STA 1 - Channels {ch_group1}')
        axes[0].set_xlabel('Time samples')
        axes[0].set_ylabel('Amplitude')
        axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes[0].legend(fontsize=8)
        
        for i, ch in enumerate(ch_group2):
            label = f'Ch {ch}' + (' (max)' if ch == best_ch2 else '')
            axes[1].plot(sta2[ch, :], linewidth=1.5 if ch == best_ch2 else 1,
                        alpha=1.0 if ch == best_ch2 else 0.6, label=label)
        axes[1].set_title(f'STA 2 - Channels {ch_group2}')
        axes[1].set_xlabel('Time samples')
        axes[1].set_ylabel('Amplitude')
        axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes[1].legend(fontsize=8)
        
        lags = np.arange(len(xcorr_normalized)) - (len(waveform1) - 1)
        axes[2].plot(lags, xcorr_normalized, 'k-', linewidth=1)
        axes[2].axvline(max_lag, color='c', linestyle='--', alpha=0.7)
        axes[2].plot(max_lag, max_xcorr, 'c*', markersize=15, label=f'Max: {max_xcorr:.3f}')
        axes[2].set_title(f'1D XCorr (max={max_xcorr:.3f} at lag={max_lag})')
        axes[2].set_xlabel('Lag (samples)')
        axes[2].set_ylabel('Correlation')
        axes[2].legend()
        
        fig.suptitle(title, fontsize=12)
        plt.tight_layout()
        plt.show()
    
    return max_xcorr, best_ch1, best_ch2


def compute_all_stas(emg_signal, spike_dict, fs=2000, window_ms=30, channels_to_remove=None, verbose=True):
    """Compute STAs for all MUs in a file."""
    stas = {}
    
    if verbose:
        logger.info(f"Computing STAs for {len(spike_dict)} MUs...")
        logger.info(f"EMG signal shape: {emg_signal.shape}")
        logger.info(f"Parameters: fs={fs} Hz, window={window_ms} ms")
        if channels_to_remove:
            logger.info(f"Removing channels: {channels_to_remove}")
    
    for mu_idx, spikes in spike_dict.items():
        sta, time_axis, n_used = calculate_spike_triggered_average(
            emg_signal, spikes, fs=fs, window_ms=window_ms
        )
        
        if channels_to_remove is not None:
            valid_channels = [ch for ch in channels_to_remove if ch < sta.shape[0]]
            if len(valid_channels) != len(channels_to_remove):
                invalid = set(channels_to_remove) - set(valid_channels)
                logger.warning(f"  MU {mu_idx}: Invalid channel indices ignored: {invalid}")
            sta = np.delete(sta, valid_channels, axis=0)
        
        stas[mu_idx] = sta
        
        if verbose:
            logger.info(f"  MU {mu_idx}: {len(spikes)} spikes, {n_used} used, STA shape: {sta.shape}")
    
    return stas


def plot_xcc_matrix(xcc_matrix, file1_mus, file2_mus, threshold=0.7):
    """Plot the full cross-correlation matrix between all MU pairs."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    im = ax.imshow(xcc_matrix, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    
    for i in range(xcc_matrix.shape[0]):
        max_j = np.argmax(xcc_matrix[i, :])
        if xcc_matrix[i, max_j] >= threshold:
            ax.plot(max_j, i, 'r*', markersize=12)
    
    ax.set_xticks(range(len(file2_mus)))
    ax.set_xticklabels(file2_mus)
    ax.set_yticks(range(len(file1_mus)))
    ax.set_yticklabels(file1_mus)
    ax.set_xlabel('MU index (File 2)')
    ax.set_ylabel('MU index (File 1)')
    ax.set_title(f'Cross-Correlation Matrix (threshold={threshold})')
    
    plt.colorbar(im, ax=ax, label='XCC')
    plt.tight_layout()
    return fig, xcc_matrix


def track_motor_units(stas_file1, stas_file2, threshold=0.7, verbose=True,
                      debug_plot_pairs=False, debug_plot_xcorr=False,
                      plot_matrix=True, plot_matched_pairs=True):
    """Track MUs between two files by comparing STA shapes."""
    results = {"MU_file1": [], "MU_file2": [], "XCC": [], "Ch_file1": [], "Ch_file2": []}
    
    file1_mus = list(stas_file1.keys())
    file2_mus = list(stas_file2.keys())
    
    if verbose:
        logger.info("=" * 60)
        logger.info("MOTOR UNIT TRACKING")
        logger.info("=" * 60)
        logger.info(f"File 1: {len(file1_mus)} MUs (indices: {file1_mus})")
        logger.info(f"File 2: {len(file2_mus)} MUs (indices: {file2_mus})")
        logger.info(f"Threshold: {threshold}")
        logger.info(f"Total comparisons: {len(file1_mus) * len(file2_mus)}")
        logger.info("-" * 60)
    
    xcc_matrix = np.zeros((len(file1_mus), len(file2_mus)))
    
    for i, (mu1_idx, sta1) in enumerate(stas_file1.items()):
        for j, (mu2_idx, sta2) in enumerate(stas_file2.items()):
            
            min_samples = min(sta1.shape[1], sta2.shape[1])
            sta1_trimmed = sta1[:, :min_samples]
            sta2_trimmed = sta2[:, :min_samples]
            
            min_channels = min(sta1_trimmed.shape[0], sta2_trimmed.shape[0])
            sta1_trimmed = sta1_trimmed[:min_channels, :]
            sta2_trimmed = sta2_trimmed[:min_channels, :]
            
            xcc, ch1, ch2 = norm_twod_xcorr(
                sta1_trimmed, sta2_trimmed, 
                debug_plot=debug_plot_xcorr,
                title=f"MU {mu1_idx} (File 1) vs MU {mu2_idx} (File 2)"
            )
            
            xcc_matrix[i, j] = xcc
            
            if xcc >= threshold:
                logger.info(f"  ✓ MU {mu1_idx} vs MU {mu2_idx}: XCC = {xcc:.4f} [MATCH] (Ch {ch1} vs Ch {ch2})")
                results["MU_file1"].append(mu1_idx)
                results["MU_file2"].append(mu2_idx)
                results["XCC"].append(xcc)
                results["Ch_file1"].append(ch1)
                results["Ch_file2"].append(ch2)
            else:
                logger.debug(f"    MU {mu1_idx} vs MU {mu2_idx}: XCC = {xcc:.4f}")
    
    if plot_matrix:
        logger.info("-" * 60)
        logger.info("Plotting XCC matrix...")
        fig, _ = plot_xcc_matrix(xcc_matrix, file1_mus, file2_mus, threshold)
        plt.show()
    
    tracking_res = pd.DataFrame(results)
    
    if verbose:
        logger.info("-" * 60)
        logger.info(f"Pairs above threshold ({threshold}): {len(tracking_res)}")
    
    if not tracking_res.empty:
        tracking_res_unfiltered = tracking_res.copy()
        tracking_res = filter_best_matches(tracking_res, verbose=verbose)
        
        if verbose:
            logger.info(f"Pairs after filtering: {len(tracking_res)}")
            logger.info("-" * 60)
            logger.info("FINAL TRACKING RESULTS:")
            logger.info(f"\n{tracking_res.to_string()}")
        
        if plot_matched_pairs and len(tracking_res) > 0:
            logger.info("-" * 60)
            logger.info("Plotting matched pairs...")
            plot_all_matched_pairs(stas_file1, stas_file2, tracking_res)
    else:
        if verbose:
            logger.warning("No matches found above threshold!")
    
    logger.info("=" * 60)
    
    return tracking_res


def filter_best_matches(tracking_res, verbose=True):
    """Keep only the highest XCC match for each MU."""
    if verbose:
        logger.info("Filtering to keep best matches only...")
    
    tracking_res = tracking_res.sort_values("XCC", ascending=False)
    
    before_file1 = len(tracking_res)
    tracking_res = tracking_res.drop_duplicates(subset="MU_file1", keep="first")
    if verbose:
        logger.info(f"  After MU_file1 dedup: {before_file1} -> {len(tracking_res)}")
    
    before_file2 = len(tracking_res)
    tracking_res = tracking_res.drop_duplicates(subset="MU_file2", keep="first")
    if verbose:
        logger.info(f"  After MU_file2 dedup: {before_file2} -> {len(tracking_res)}")
    
    return tracking_res.sort_values("MU_file1").reset_index(drop=True)


def plot_all_matched_pairs(stas_file1, stas_file2, tracking_res):
    """Plot matched pairs showing only channels used in comparison."""
    n_pairs = len(tracking_res)

    if n_pairs == 0:
        logger.warning("No pairs to plot!")
        return

    def get_channel_group(best_ch, n_ch):
        channels = [best_ch]
        if best_ch > 0:
            channels.insert(0, best_ch - 1)
        if best_ch < n_ch - 1:
            channels.append(best_ch + 1)
        return channels

    for _, row in tracking_res.iterrows():
        mu1 = row['MU_file1']
        mu2 = row['MU_file2']
        xcc = row['XCC']
        ch1 = int(row['Ch_file1'])
        ch2 = int(row['Ch_file2'])

        sta1 = stas_file1[mu1]
        sta2 = stas_file2[mu2]

        min_t = min(sta1.shape[1], sta2.shape[1])

        ch_group1 = get_channel_group(ch1, sta1.shape[0])
        ch_group2 = get_channel_group(ch2, sta2.shape[0])

        fig = plt.figure(figsize=(14, 8))

        # Top: Max amplitude channels overlaid
        ax_top = fig.add_subplot(2, 1, 1)
        time_axis = np.arange(min_t)
        ax_top.plot(time_axis, sta1[ch1, :min_t], 'b-', linewidth=2, label=f'File 1 Ch {ch1}')
        ax_top.plot(time_axis, sta2[ch2, :min_t], 'r--', linewidth=2, label=f'File 2 Ch {ch2}')
        ax_top.axhline(0, color='gray', linestyle=':', alpha=0.5)
        ax_top.set_xlabel('Time samples')
        ax_top.set_ylabel('Amplitude')
        ax_top.set_title(f'Max Amplitude Channels Comparison')
        ax_top.legend()

        # Bottom: All channels used in comparison
        n_ch_show = max(len(ch_group1), len(ch_group2))

        for i in range(n_ch_show):
            # File 1 channels
            ax_left = fig.add_subplot(2, n_ch_show * 2, n_ch_show * 2 + 1 + i)
            if i < len(ch_group1):
                ch = ch_group1[i]
                color = 'b' if ch == ch1 else 'cornflowerblue'
                lw = 2 if ch == ch1 else 1
                ax_left.plot(sta1[ch, :min_t], color=color, linewidth=lw)
                ax_left.set_title(f'F1 Ch {ch}' + (' (max)' if ch == ch1 else ''), fontsize=9)
            ax_left.axhline(0, color='gray', linestyle=':', alpha=0.3)
            ax_left.set_xticks([])
            ax_left.set_yticks([])

            # File 2 channels
            ax_right = fig.add_subplot(2, n_ch_show * 2, n_ch_show * 2 + n_ch_show + 1 + i)
            if i < len(ch_group2):
                ch = ch_group2[i]
                color = 'r' if ch == ch2 else 'lightcoral'
                lw = 2 if ch == ch2 else 1
                ax_right.plot(sta2[ch, :min_t], color=color, linewidth=lw)
                ax_right.set_title(f'F2 Ch {ch}' + (' (max)' if ch == ch2 else ''), fontsize=9)
            ax_right.axhline(0, color='gray', linestyle=':', alpha=0.3)
            ax_right.set_xticks([])
            ax_right.set_yticks([])

        fig.suptitle(f'MU {mu1} (File 1) vs MU {mu2} (File 2) | XCC = {xcc:.3f}', fontsize=12)
        plt.tight_layout()
        plt.show()


def set_debug_level(level='INFO'):
    """Set logging level: 'DEBUG', 'INFO', 'WARNING', 'ERROR'"""
    logger.setLevel(getattr(logging, level))
    logging.getLogger().setLevel(getattr(logging, level))


def remove_channels(emg_signal, channels_to_remove):
    """Remove specified channels from EMG signal."""
    n_channels = emg_signal.shape[0]
    valid_channels = [ch for ch in channels_to_remove if 0 <= ch < n_channels]
    kept_channels = [i for i in range(n_channels) if i not in valid_channels]
    channel_mapping = {new_idx: orig_idx for new_idx, orig_idx in enumerate(kept_channels)}
    
    cleaned_emg = np.delete(emg_signal, valid_channels, axis=0)
    
    logger.info(f"Removed channels {valid_channels}: {n_channels} -> {cleaned_emg.shape[0]} channels")
    logger.info(f"Channel mapping (new -> original): {channel_mapping}")
    
    return cleaned_emg, channel_mapping

In [ ]:
from scipy.signal import correlate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def norm_twod_xcorr(sta1, sta2, debug_plot=False, title=""):
    """
    Compute normalized 1D cross-correlation between two STAs using the channel
    with greatest amplitude and its immediate neighbors from each STA.
    """
    n_channels = sta1.shape[0]
    
    # Find channel with greatest amplitude (peak-to-peak) for each STA
    amp1 = np.ptp(sta1, axis=1)
    amp2 = np.ptp(sta2, axis=1)
    
    best_ch1 = np.argmax(amp1)
    best_ch2 = np.argmax(amp2)
    
    def get_channel_group(best_ch, n_ch):
        channels = [best_ch]
        if best_ch > 0:
            channels.insert(0, best_ch - 1)
        if best_ch < n_ch - 1:
            channels.append(best_ch + 1)
        return channels
    
    ch_group1 = get_channel_group(best_ch1, n_channels)
    ch_group2 = get_channel_group(best_ch2, sta2.shape[0])
    
    waveform1 = sta1[ch_group1, :].flatten()
    waveform2 = sta2[ch_group2, :].flatten()
    
    logger.debug(f"STA1 best channel: {best_ch1}, using channels {ch_group1}")
    logger.debug(f"STA2 best channel: {best_ch2}, using channels {ch_group2}")
    
    wf1_norm = (waveform1 - np.mean(waveform1)) / (np.std(waveform1) + 1e-10)
    wf2_norm = (waveform2 - np.mean(waveform2)) / (np.std(waveform2) + 1e-10)
    
    xcorr = correlate(wf1_norm, wf2_norm, mode='full')
    xcorr_normalized = xcorr / len(waveform1)
    
    max_xcorr = np.max(xcorr_normalized)
    max_lag = np.argmax(xcorr_normalized) - (len(waveform1) - 1)
    
    logger.debug(f"XCorr max: {max_xcorr:.4f} at lag {max_lag}")
    
    if debug_plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        for i, ch in enumerate(ch_group1):
            label = f'Ch {ch}' + (' (max)' if ch == best_ch1 else '')
            axes[0].plot(sta1[ch, :], linewidth=1.5 if ch == best_ch1 else 1, 
                        alpha=1.0 if ch == best_ch1 else 0.6, label=label)
        axes[0].set_title(f'STA 1 - Channels {ch_group1}')
        axes[0].set_xlabel('Time samples')
        axes[0].set_ylabel('Amplitude')
        axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes[0].legend(fontsize=8)
        
        for i, ch in enumerate(ch_group2):
            label = f'Ch {ch}' + (' (max)' if ch == best_ch2 else '')
            axes[1].plot(sta2[ch, :], linewidth=1.5 if ch == best_ch2 else 1,
                        alpha=1.0 if ch == best_ch2 else 0.6, label=label)
        axes[1].set_title(f'STA 2 - Channels {ch_group2}')
        axes[1].set_xlabel('Time samples')
        axes[1].set_ylabel('Amplitude')
        axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes[1].legend(fontsize=8)
        
        lags = np.arange(len(xcorr_normalized)) - (len(waveform1) - 1)
        axes[2].plot(lags, xcorr_normalized, 'k-', linewidth=1)
        axes[2].axvline(max_lag, color='c', linestyle='--', alpha=0.7)
        axes[2].plot(max_lag, max_xcorr, 'c*', markersize=15, label=f'Max: {max_xcorr:.3f}')
        axes[2].set_title(f'1D XCorr (max={max_xcorr:.3f} at lag={max_lag})')
        axes[2].set_xlabel('Lag (samples)')
        axes[2].set_ylabel('Correlation')
        axes[2].legend()
        
        fig.suptitle(title, fontsize=12)
        plt.tight_layout()
        plt.show()
    
    return max_xcorr, best_ch1, best_ch2


def compute_all_stas(emg_signal, spike_dict, fs=2000, window_ms=30, channels_to_remove=None, verbose=True):
    """Compute STAs for all MUs in a file."""
    stas = {}
    
    if verbose:
        logger.info(f"Computing STAs for {len(spike_dict)} MUs...")
        logger.info(f"EMG signal shape: {emg_signal.shape}")
        logger.info(f"Parameters: fs={fs} Hz, window={window_ms} ms")
        if channels_to_remove:
            logger.info(f"Removing channels: {channels_to_remove}")
    
    for mu_idx, spikes in spike_dict.items():
        sta, time_axis, n_used = calculate_spike_triggered_average(
            emg_signal, spikes, fs=fs, window_ms=window_ms
        )
        
        if channels_to_remove is not None:
            valid_channels = [ch for ch in channels_to_remove if ch < sta.shape[0]]
            if len(valid_channels) != len(channels_to_remove):
                invalid = set(channels_to_remove) - set(valid_channels)
                logger.warning(f"  MU {mu_idx}: Invalid channel indices ignored: {invalid}")
            sta = np.delete(sta, valid_channels, axis=0)
        
        stas[mu_idx] = sta
        
        if verbose:
            logger.info(f"  MU {mu_idx}: {len(spikes)} spikes, {n_used} used, STA shape: {sta.shape}")
    
    return stas


def plot_xcc_matrix(xcc_matrix, file1_mus, file2_mus, threshold=0.7):
    """Plot the full cross-correlation matrix between all MU pairs."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    im = ax.imshow(xcc_matrix, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    
    for i in range(xcc_matrix.shape[0]):
        max_j = np.argmax(xcc_matrix[i, :])
        if xcc_matrix[i, max_j] >= threshold:
            ax.plot(max_j, i, 'r*', markersize=12)
    
    ax.set_xticks(range(len(file2_mus)))
    ax.set_xticklabels(file2_mus)
    ax.set_yticks(range(len(file1_mus)))
    ax.set_yticklabels(file1_mus)
    ax.set_xlabel('MU index (File 2)')
    ax.set_ylabel('MU index (File 1)')
    ax.set_title(f'Cross-Correlation Matrix (threshold={threshold})')
    
    plt.colorbar(im, ax=ax, label='XCC')
    plt.tight_layout()
    return fig, xcc_matrix


def track_motor_units(stas_file1, stas_file2, threshold=0.7, verbose=True,
                      debug_plot_pairs=False, debug_plot_xcorr=False,
                      plot_matrix=True, plot_matched_pairs=True):
    """Track MUs between two files by comparing STA shapes."""
    results = {"MU_file1": [], "MU_file2": [], "XCC": [], "Ch_file1": [], "Ch_file2": []}
    
    file1_mus = list(stas_file1.keys())
    file2_mus = list(stas_file2.keys())
    
    if verbose:
        logger.info("=" * 60)
        logger.info("MOTOR UNIT TRACKING")
        logger.info("=" * 60)
        logger.info(f"File 1: {len(file1_mus)} MUs (indices: {file1_mus})")
        logger.info(f"File 2: {len(file2_mus)} MUs (indices: {file2_mus})")
        logger.info(f"Threshold: {threshold}")
        logger.info(f"Total comparisons: {len(file1_mus) * len(file2_mus)}")
        logger.info("-" * 60)
    
    xcc_matrix = np.zeros((len(file1_mus), len(file2_mus)))
    
    for i, (mu1_idx, sta1) in enumerate(stas_file1.items()):
        for j, (mu2_idx, sta2) in enumerate(stas_file2.items()):
            
            min_samples = min(sta1.shape[1], sta2.shape[1])
            sta1_trimmed = sta1[:, :min_samples]
            sta2_trimmed = sta2[:, :min_samples]
            
            min_channels = min(sta1_trimmed.shape[0], sta2_trimmed.shape[0])
            sta1_trimmed = sta1_trimmed[:min_channels, :]
            sta2_trimmed = sta2_trimmed[:min_channels, :]
            
            xcc, ch1, ch2 = norm_twod_xcorr(
                sta1_trimmed, sta2_trimmed, 
                debug_plot=debug_plot_xcorr,
                title=f"MU {mu1_idx} (File 1) vs MU {mu2_idx} (File 2)"
            )
            
            xcc_matrix[i, j] = xcc
            
            if xcc >= threshold:
                logger.info(f"  ✓ MU {mu1_idx} vs MU {mu2_idx}: XCC = {xcc:.4f} [MATCH] (Ch {ch1} vs Ch {ch2})")
                results["MU_file1"].append(mu1_idx)
                results["MU_file2"].append(mu2_idx)
                results["XCC"].append(xcc)
                results["Ch_file1"].append(ch1)
                results["Ch_file2"].append(ch2)
            else:
                logger.debug(f"    MU {mu1_idx} vs MU {mu2_idx}: XCC = {xcc:.4f}")
    
    if plot_matrix:
        logger.info("-" * 60)
        logger.info("Plotting XCC matrix...")
        fig, _ = plot_xcc_matrix(xcc_matrix, file1_mus, file2_mus, threshold)
        plt.show()
    
    tracking_res = pd.DataFrame(results)
    
    if verbose:
        logger.info("-" * 60)
        logger.info(f"Pairs above threshold ({threshold}): {len(tracking_res)}")
    
    if not tracking_res.empty:
        tracking_res_unfiltered = tracking_res.copy()
        tracking_res = filter_best_matches(tracking_res, verbose=verbose)
        
        if verbose:
            logger.info(f"Pairs after filtering: {len(tracking_res)}")
            logger.info("-" * 60)
            logger.info("FINAL TRACKING RESULTS:")
            logger.info(f"\n{tracking_res.to_string()}")
        
        if plot_matched_pairs and len(tracking_res) > 0:
            logger.info("-" * 60)
            logger.info("Plotting matched pairs...")
            plot_all_matched_pairs(stas_file1, stas_file2, tracking_res)
    else:
        if verbose:
            logger.warning("No matches found above threshold!")
    
    logger.info("=" * 60)
    
    return tracking_res


def filter_best_matches(tracking_res, verbose=True):
    """Keep only the highest XCC match for each MU."""
    if verbose:
        logger.info("Filtering to keep best matches only...")
    
    tracking_res = tracking_res.sort_values("XCC", ascending=False)
    
    before_file1 = len(tracking_res)
    tracking_res = tracking_res.drop_duplicates(subset="MU_file1", keep="first")
    if verbose:
        logger.info(f"  After MU_file1 dedup: {before_file1} -> {len(tracking_res)}")
    
    before_file2 = len(tracking_res)
    tracking_res = tracking_res.drop_duplicates(subset="MU_file2", keep="first")
    if verbose:
        logger.info(f"  After MU_file2 dedup: {before_file2} -> {len(tracking_res)}")
    
    return tracking_res.sort_values("MU_file1").reset_index(drop=True)


def plot_all_matched_pairs(stas_file1, stas_file2, tracking_res):
    """Plot matched pairs showing only channels used in comparison."""
    n_pairs = len(tracking_res)

    if n_pairs == 0:
        logger.warning("No pairs to plot!")
        return

    def get_channel_group(best_ch, n_ch):
        channels = [best_ch]
        if best_ch > 0:
            channels.insert(0, best_ch - 1)
        if best_ch < n_ch - 1:
            channels.append(best_ch + 1)
        return channels

    for _, row in tracking_res.iterrows():
        mu1 = row['MU_file1']
        mu2 = row['MU_file2']
        xcc = row['XCC']
        ch1 = int(row['Ch_file1'])
        ch2 = int(row['Ch_file2'])

        sta1 = stas_file1[mu1]
        sta2 = stas_file2[mu2]

        min_t = min(sta1.shape[1], sta2.shape[1])

        ch_group1 = get_channel_group(ch1, sta1.shape[0])
        ch_group2 = get_channel_group(ch2, sta2.shape[0])

        fig = plt.figure(figsize=(14, 8))

        # Top: Max amplitude channels overlaid
        ax_top = fig.add_subplot(2, 1, 1)
        time_axis = np.arange(min_t)
        ax_top.plot(time_axis, sta1[ch1, :min_t], 'b-', linewidth=2, label=f'File 1 Ch {ch1}')
        ax_top.plot(time_axis, sta2[ch2, :min_t], 'r--', linewidth=2, label=f'File 2 Ch {ch2}')
        ax_top.axhline(0, color='gray', linestyle=':', alpha=0.5)
        ax_top.set_xlabel('Time samples')
        ax_top.set_ylabel('Amplitude')
        ax_top.set_title(f'Max Amplitude Channels Comparison')
        ax_top.legend()

        # Bottom: All channels used in comparison
        n_ch_show = max(len(ch_group1), len(ch_group2))

        for i in range(n_ch_show):
            # File 1 channels
            ax_left = fig.add_subplot(2, n_ch_show * 2, n_ch_show * 2 + 1 + i)
            if i < len(ch_group1):
                ch = ch_group1[i]
                color = 'b' if ch == ch1 else 'cornflowerblue'
                lw = 2 if ch == ch1 else 1
                ax_left.plot(sta1[ch, :min_t], color=color, linewidth=lw)
                ax_left.set_title(f'F1 Ch {ch}' + (' (max)' if ch == ch1 else ''), fontsize=9)
            ax_left.axhline(0, color='gray', linestyle=':', alpha=0.3)
            ax_left.set_xticks([])
            ax_left.set_yticks([])

            # File 2 channels
            ax_right = fig.add_subplot(2, n_ch_show * 2, n_ch_show * 2 + n_ch_show + 1 + i)
            if i < len(ch_group2):
                ch = ch_group2[i]
                color = 'r' if ch == ch2 else 'lightcoral'
                lw = 2 if ch == ch2 else 1
                ax_right.plot(sta2[ch, :min_t], color=color, linewidth=lw)
                ax_right.set_title(f'F2 Ch {ch}' + (' (max)' if ch == ch2 else ''), fontsize=9)
            ax_right.axhline(0, color='gray', linestyle=':', alpha=0.3)
            ax_right.set_xticks([])
            ax_right.set_yticks([])

        fig.suptitle(f'MU {mu1} (File 1) vs MU {mu2} (File 2) | XCC = {xcc:.3f}', fontsize=12)
        plt.tight_layout()
        plt.show()


def set_debug_level(level='INFO'):
    """Set logging level: 'DEBUG', 'INFO', 'WARNING', 'ERROR'"""
    logger.setLevel(getattr(logging, level))
    logging.getLogger().setLevel(getattr(logging, level))


def remove_channels(emg_signal, channels_to_remove):
    """Remove specified channels from EMG signal."""
    n_channels = emg_signal.shape[0]
    valid_channels = [ch for ch in channels_to_remove if 0 <= ch < n_channels]
    kept_channels = [i for i in range(n_channels) if i not in valid_channels]
    channel_mapping = {new_idx: orig_idx for new_idx, orig_idx in enumerate(kept_channels)}
    
    cleaned_emg = np.delete(emg_signal, valid_channels, axis=0)
    
    logger.info(f"Removed channels {valid_channels}: {n_channels} -> {cleaned_emg.shape[0]} channels")
    logger.info(f"Channel mapping (new -> original): {channel_mapping}")
    
    return cleaned_emg, channel_mapping

In [ ]:
def plot_matched_pair(stas1, stas2, mu1_idx, mu2_idx, xcc):
    """Plot STAs side by side for a matched pair."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    sta1 = stas1[mu1_idx]
    sta2 = stas2[mu2_idx]
    
    # Plot as heatmaps for easy comparison
    im1 = axes[0].imshow(sta1, aspect='auto', cmap='RdBu_r')
    axes[0].set_title(f'File 1 - MU {mu1_idx}')
    axes[0].set_xlabel('Time samples')
    axes[0].set_ylabel('Channel')
    plt.colorbar(im1, ax=axes[0])
    
    im2 = axes[1].imshow(sta2, aspect='auto', cmap='RdBu_r')
    axes[1].set_title(f'File 2 - MU {mu2_idx}')
    axes[1].set_xlabel('Time samples')
    axes[1].set_ylabel('Channel')
    plt.colorbar(im2, ax=axes[1])
    
    fig.suptitle(f'Matched Pair: XCC = {xcc:.3f}')
    plt.tight_layout()
    plt.show()

In [ ]:
stas_dict_full.keys()

In [ ]:
stas1 = {idx: info['sta'] for idx, info in stas_dict_full.items()}
stas2 = {idx: info['sta'] for idx, info in stas_dict.items()}

tracking_results = track_motor_units(
    stas_file1=stas1,
    stas_file2=stas2)


def plot_matched_pair(stas1, stas2, mu1_idx, mu2_idx, xcc):
    """Plot STAs side by side for a matched pair."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    sta1 = stas1[mu1_idx]
    sta2 = stas2[mu2_idx]
    
    # Plot as heatmaps for easy comparison
    im1 = axes[0].imshow(sta1, aspect='auto', cmap='RdBu_r')
    axes[0].set_title(f'File 1 - MU {mu1_idx}')
    axes[0].set_xlabel('Time samples')
    axes[0].set_ylabel('Channel')
    plt.colorbar(im1, ax=axes[0])
    
    im2 = axes[1].imshow(sta2, aspect='auto', cmap='RdBu_r')
    axes[1].set_title(f'File 2 - MU {mu2_idx}')
    axes[1].set_xlabel('Time samples')
    axes[1].set_ylabel('Channel')
    plt.colorbar(im2, ax=axes[1])
    
    fig.suptitle(f'Matched Pair: XCC = {xcc:.3f}')
    plt.tight_layout()
    plt.show()

# Plot all matched pairs
for _, row in tracking_results.iterrows():
    try:
        plot_matched_pair(
            stas_dict_full, stas_dict,
            row['MU_file1'], row['MU_file2'], row['XCC']
        )
    except Exception as e:
        logger.error(f"Error plotting MU {row['MU_file1']} vs MU {row['MU_file2']}: {e}")



# Online control

In [ ]:
import pickle
import numpy as np
import os 
import sys
import json
import matplotlib.pyplot as plt

In [ ]:
#search all folders in a path
import os
def find_folders_with_keyword(base_path, keyword):
    """Search for folders containing a specific keyword in their name."""
    matching_folders = []
    for root, dirs, files in os.walk(base_path):
        for dir_name in dirs:
            if keyword in dir_name:
                matching_folders.append(os.path.join(root, dir_name))
    return matching_folders

folders = find_folders_with_keyword(r"C:\Users\velar\SynologyDrive\Personal\ExperimentalData\0602026Session1", "emg_recording")

folders = folders[-2:]  

# folders = folders[0:2]  


In [ ]:
import numpy as np
from scipy.interpolate import interp1d

def remove_spikes(emg, threshold_std=5):
    """
    Remove large spikes via simple thresholding and linear interpolation.
    
    Parameters
    ----------
    emg : ndarray, shape (n_channels, n_samples)
    threshold_std : float
        Number of standard deviations for threshold
    
    Returns
    -------
    emg_clean : ndarray
    spike_mask : ndarray, bool
    """
    emg_clean = emg.copy()
    spike_mask = np.zeros_like(emg, dtype=bool)
    
    for ch in range(emg.shape[0]):
        signal = emg[ch]
        threshold = threshold_std * np.std(signal)
        
        spikes = np.abs(signal) > threshold
        spike_mask[ch] = spikes
        
        if np.any(spikes):
            clean_idx = np.where(~spikes)[0]
            spike_idx = np.where(spikes)[0]
            
            if len(clean_idx) > 1:
                interp = interp1d(clean_idx, signal[clean_idx], 
                                  kind='linear', bounds_error=False, 
                                  fill_value='extrapolate')
                emg_clean[ch, spike_idx] = interp(spike_idx)
    
    return emg_clean, spike_mask


In [ ]:
decomp_combined = {}
decomp_all = {}


n_spikes_all = 0
isi_all = []
cov_all = []

n_spikes_combined = 0
isi_combined = []
cov_combined = []

n_reps_mu = 8

for folder in folders:
    decomp_path_all = os.path.join(folder,f"processed_{n_reps_mu}reps", "full_emg_data.pkl")
    decomp_path_combined = os.path.join(folder,f"processed_{n_reps_mu}reps", "combined_emg_data.pkl")
    if os.path.exists(decomp_path_all):
        with open(decomp_path_all, 'rb') as f:
            decomp_data, _ = pickle.load(f)
            n_spikes_all += len(decomp_data['spikes'].keys())
            for i, spikes in decomp_data['spikes'].items():
                isi = np.diff(spikes / 2000)
                cov = np.std(isi) / np.mean(isi)
                isi_all.extend(isi)
                cov_all.append(cov)

            decomp_all[folder] = decomp_data
    else:
        print(f"Decomposition file not found: {decomp_path_all}")
        
    if os.path.exists(decomp_path_combined):
        with open(decomp_path_combined, 'rb') as f:
            decomp_data, _ = pickle.load(f)
            n_spikes_combined += len(decomp_data['spikes'].keys())
            for i, spikes in decomp_data['spikes'].items():
                isi = np.diff(spikes / 2000)
                cov = np.std(isi) / np.mean(isi)
                isi_combined.extend(isi)
                cov_combined.append(cov)

            decomp_combined[folder] = decomp_data
    else:
        print(f"Decomposition file not found: {decomp_path_combined}")

mean_spikes_per_file_all = n_spikes_all / len(decomp_all) if decomp_all else 0
mean_spikes_per_file_combined = n_spikes_combined / len(decomp_combined) if decomp_combined else 0

std_spikes_per_file_all = np.std([len(data['spikes'].keys()) for data in decomp_all.values()]) if decomp_all else 0
std_spikes_per_file_combined = np.std([len(data['spikes'].keys()) for data in decomp_combined.values()]) if decomp_combined else 0

print(f"Total spikes (all): {n_spikes_all}, Mean per file: {mean_spikes_per_file_all:.2f} ± {std_spikes_per_file_all:.2f}")
print(f"Total spikes (combined): {n_spikes_combined}, Mean per file: {mean_spikes_per_file_combined:.2f} ± {std_spikes_per_file_combined:.2f}")
print(f"Total ISI (all): {len(isi_all)}, Mean ISI: {np.mean(isi_all):.2f} ± {np.std(isi_all):.2f}")
print(f"Total ISI (combined): {len(isi_combined)}, Mean ISI: {np.mean(isi_combined):.2f} ± {np.std(isi_combined):.2f}")
print(f"Mean CoV of ISI (all): {np.mean(cov_all):.2f} ± {np.std(cov_all):.2f}, MAX CoV: {np.max(cov_all):.2f}, MIN CoV: {np.min(cov_all):.2f}")
print(f"Mean CoV of ISI (combined): {np.mean(cov_combined):.2f} ± {np.std(cov_combined):.2f}, MAX CoV: {np.max(cov_combined):.2f}, MIN CoV: {np.min(cov_combined):.2f}")

In [ ]:
# combined_good = [10,6,5,4]
combined_good = [11,5,4,7]
all_good = [8,10,6,3]

mean_combined_good = np.mean(combined_good)
std_combined_good = np.std(combined_good)

mean_all_good = np.mean(all_good)
std_all_good = np.std(all_good)

print(f"Combined good MUs: {mean_combined_good:.2f} ± {std_combined_good:.2f}")
print(f"All good MUs: {mean_all_good:.2f} ± {std_all_good:.2f}")


In [ ]:


print(decomp_combined.keys())

print(decomp_combined[folders[0]].keys())

In [ ]:
# --- Step 1: Stack MU filters and re-index centroids ---

stacked_mu_filters = None
stacked_centroids = {}
mu_offset = 0

for folder, result in decomp_combined.items():
    n_mus = result['mu_filters'].shape[1]
    print(f"{os.path.basename(folder)}: {n_mus} MUs, filters shape {result['mu_filters'].shape}")
    
    # Stack filters horizontally
    if stacked_mu_filters is None:
        stacked_mu_filters = result['mu_filters']
    else:
        stacked_mu_filters = np.hstack((stacked_mu_filters, result['mu_filters']))
    
    # Re-index centroids with offset
    if result.get('centroids') is not None:
        for mu_idx, centroid_values in result['centroids'].items():
            new_idx = int(mu_idx) + mu_offset
            stacked_centroids[new_idx] = centroid_values
    
    mu_offset += n_mus

print(f"\nStacked filters shape: {stacked_mu_filters.shape}")
print(f"Total MUs: {mu_offset}")
print(f"Centroids re-indexed: {list(stacked_centroids.keys())}")

In [ ]:
# --- Step 2: Load and concatenate EMG data from all folders ---

recorded_trials = []
recorded_onemove = []

concatenated_emg = None 


for folder in decomp_combined.keys():
    calibration_path = os.path.join(folder, f"processed_{n_reps_mu}reps", "concatenated_emg_channels.pkl")
    emg_path = os.path.join(folder, f"processed_base", "concatenated_emg_channels.pkl")
    onemove_path = os.path.join(folder, "move_001.pkl")
    
    with open(calibration_path, 'rb') as f:
        file = pickle.load(f)
        emg_data = file["emg_data"]
        # emg_pause = file["emg_pause"]
        print(emg_data)
        emg_data_clean, _ = remove_spikes(emg_data, threshold_std=8)
        # emg_pause_clean, _ = remove_spikes(emg_pause, threshold_std=8)
        temp = {"data": emg_data_clean}

        # concatenated_emg = emg_pause_clean if concatenated_emg is None else np.hstack((concatenated_emg, emg_pause_clean))
        # concatenated_emg, _  = remove_spikes(concatenated_emg, threshold_std=8) 
        recorded_trials.append(temp)

    with open(emg_path, 'rb') as f:
        file = pickle.load(f)

        emg_pause = file["emg_pause"]
        emg_pause_clean, _ = remove_spikes(emg_pause, threshold_std=8)

        concatenated_emg = emg_pause_clean if concatenated_emg is None else np.hstack((concatenated_emg, emg_pause_clean))
        # concatenated_emg, _  = remove_spikes(concatenated_emg, threshold_std=8) 

    with open(onemove_path, 'rb') as f:
        onemove_data = pickle.load(f)
        emg_onemove = onemove_data["data"]
        emg_onemove = np.delete(emg_onemove, [0, 1,2,3], axis=0)  # Remove first four channels
        emg_onemove_clean, _ = remove_spikes(emg_onemove, threshold_std=8)
        onemove_temp = {"data": emg_onemove_clean}
        recorded_onemove.append(onemove_temp)




In [ ]:
import os 
import sys

sys.path.append(r"C:\Users\velar\Documents\GitHub\PatientGUI")

from legacy_pipeline.decomposition import compute_combined_whitening_and_centroids



In [ ]:
plt.plot(recorded_trials[0]["data"][0, :])

In [ ]:
# --- Step 4: Recalibrate filters via Spike-Triggered Averaging (Farina 2025) ---
# This reintroduces temporal overlaps removed during offline peel-off,
# making filters suitable for online (non-peel-off) spike detection.

selected_indices = [0,1]
Z_combined, recalibrated_filters, recalibrated_centroids, norm_factors,\
recalibrated_sil, recalibrated_spikes, sources,\
stacked_mu_filters = compute_combined_whitening_and_centroids(recorded_trials=recorded_trials, selected_indices=selected_indices, stacked_mu_filters=stacked_mu_filters, fsamp = 2000, config_path=r"C:\Users\velar\SynologyDrive\Personal\Thesis Code\src\configs\cbss.json")

In [ ]:
print(recalibrated_centroids)

In [ ]:
from muniverse.algorithms.core import est_spike_times, extension

recalibrated_spikes = {}
recalibrated_sources = {}
all_recalibrated_sources = {}

# Whiten full signal
extended_sig = extension(Y=concatenated_emg, R=16)
white_sig = Z_combined @ extended_sig  # (n_channels, n_samples)

# --- Windowed source computation + spike detection (100 ms windows) ---
fsamp = 2000
win_samples = int(0.1 * fsamp)  # 200 samples
n_total = white_sig.shape[1]
n_windows = int(np.ceil(n_total / win_samples))
n_mus = recalibrated_filters.shape[1]

for i in range(n_mus):
    all_spikes = []
    all_sources = []
    offset,scale = norm_factors[i]  
    for w in range(n_windows):
        start = w * win_samples
        end = min(start + win_samples, n_total)
        white_chunk = white_sig[:, start:end]

        source_chunk = recalibrated_filters[:, i] @ white_chunk  # (n_samples,)

        # offset = np.median(source_chunk)
        # scale = np.percentile(source_chunk, 99) - offset


        if scale > 0:
            source_chunk = (source_chunk - offset) / scale
        source_chunk = np.clip(source_chunk, 0, 1)

        spikes_w, _, _ = est_spike_times(
            source_chunk, fsamp=fsamp, cluster="centroid",
            centroids=recalibrated_centroids[i]
        )
        all_spikes.extend(spikes_w + start)
        all_sources.extend(source_chunk)

    recalibrated_spikes[i] = np.array(all_spikes)
    recalibrated_sources[i] = np.array(all_sources)

    all_sources_full = recalibrated_filters[:, i] @ white_sig

    if scale > 0:
        all_sources_full = (all_sources_full - offset) / scale
    all_sources_full = np.clip(all_sources_full, 0, 1)

    all_recalibrated_sources[i] = all_sources_full



In [ ]:
# for i in range(n_mus):
#     plt.plot(all_recalibrated_sources[i], label=f'MU {i}')
#     plt.axhline(max(recalibrated_centroids[i]), color='r', linestyle='--', label='Spike Centroid')
#     plt.axhline(min(recalibrated_centroids[i]), color='c', linestyle='--', label='Noise Centroid')
#     plt.eventplot(recalibrated_spikes[i], lineoffsets=0.5, colors='k', label='Detected Spikes')
#     plt.title(f'MU {i} - Recalibrated Source and Detected Spikes')
#     print("Spike centroid:", recalibrated_centroids[i])
#     print("Noise centroid:", np.argmin(recalibrated_centroids[i]))
#     plt.show()
#     plt.close()
#     plt.plot(recalibrated_sources[i], label=f'MU {i} (windowed)')
#     plt.axhline(max(recalibrated_centroids[i]), color='r', linestyle='--', label='Spike Centroid')
#     plt.axhline(min(recalibrated_centroids[i]), color='c', linestyle='--', label='Noise Centroid')
#     plt.eventplot(recalibrated_spikes[i], lineoffsets=0.5, colors='k', label='Detected Spikes')

#     plt.title(f'MU {i} - Recalibrated Source and Detected Spikes')
#     plt.show()

In [ ]:
# --- Step 6: Package and save combined model ---

fsamp = 2000

OUTPUT_FOLDER_ONLINE = fr"C:\Users\velar\SynologyDrive\Personal\ExperimentalData\0602026Session1\online\processed_{n_reps_mu}reps"
#check if folder exists, if not create it
if not os.path.exists(OUTPUT_FOLDER_ONLINE):
    os.makedirs(OUTPUT_FOLDER_ONLINE)

combined_results = {
    "sources": sources,
    "spikes": recalibrated_spikes,
    "silhouette": recalibrated_sil,
    "mu_filters": recalibrated_filters,            # STA-recalibrated filters
    "mu_filters_original": stacked_mu_filters,     # original stacked filters
    "Z": Z_combined,                               # unified whitening matrix
    "centroids": recalibrated_centroids,
    "norm_factors": norm_factors,
}

combined_metadata = {
    "n_mus": n_mus,
    "fsamp": fsamp,
    "source_folders": list(decomp_combined.keys()),
}

output_path = os.path.join(OUTPUT_FOLDER_ONLINE, "combined_multi_trial_model_8_23.pkl")
with open(output_path, 'wb') as f:
    pickle.dump((combined_results, combined_metadata), f)

print(f"Saved combined model to: {output_path}")
print(f"  Recalibrated filters: {recalibrated_filters.shape}")
print(f"  Whitening matrix: {Z_combined.shape}")
print(f"  Total MUs: {n_mus}")
print(f"  Centroids: {len(recalibrated_centroids)}")
print(f"  SIL scores: {np.round(recalibrated_sil, 3)}")

In [ ]:
print(concatenated_emg.shape)

print(time_s.shape)

print(sources[0].shape)

In [ ]:
# --- Step 7: Raster plot (pipeline style) — EMG on top, spike rasters below ---
fsamp = 2000

sources = recalibrated_sources


mus_per_plot = 4
n_plots = int(np.ceil(n_mus / mus_per_plot))
time_s = np.arange(sources[0].shape[0]) / fsamp
duration = sources[0].shape[0] / fsamp
colors = plt.cm.tab10(np.linspace(0, 1, 10))

# Pick one EMG channel to display per plot (cycle through first few channels)
n_emg_channels = concatenated_emg.shape[0]

for plot_idx in range(n_plots):
    mu_start = plot_idx * mus_per_plot
    mu_end = min(mu_start + mus_per_plot, n_mus)
    mu_indices = list(range(mu_start, mu_end))
    n_units = len(mu_indices)
    emg_ch = plot_idx % n_emg_channels  # rotate EMG channel across plots

    fig, axes = plt.subplots(
        n_units + 1, 1,
        figsize=(14, 2 + 1.5 * n_units),
        sharex=True,
        gridspec_kw={'height_ratios': [2] + [1] * n_units}
    )

    # Top panel: raw EMG channel
    axes[0].plot(time_s, concatenated_emg[emg_ch, :], 'k-', linewidth=0.5, alpha=0.8)
    axes[0].set_ylabel(f'EMG Ch {emg_ch}', fontsize=10)
    axes[0].set_title(f'Motor Unit Raster Plot (Units {mu_start}-{mu_end - 1})', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3)

    # Raster rows
    for i, mu_idx in enumerate(mu_indices):
        ax = axes[i + 1]
        color = colors[mu_idx % len(colors)]

        spk = recalibrated_spikes[mu_idx]
        if len(spk) > 0:
            spk_times = spk / fsamp
            ax.scatter(
                spk_times, np.zeros_like(spk_times),
                c=[color], marker='|', s=100, linewidths=2, label=f'MU {mu_idx}'
            )

        ax.set_ylabel(f'MU {mu_idx}', fontsize=10, color=color)
        ax.set_ylim(-0.5, 0.5)
        ax.set_yticks([])
        ax.grid(True, alpha=0.3, axis='x')

        n_spk = len(spk)
        avg_fr = n_spk / duration if duration > 0 else 0
        sil = recalibrated_sil[mu_idx]
        # ax.text(
        #     0.98, 0.5,
        #     f'{n_spk} spikes ({avg_fr:.1f} Hz)  SIL={sil:.2f}',
        #     transform=ax.transAxes, ha='right', va='center',
        #     fontsize=9, color=color,
        #     bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8)
        # )

    axes[-1].set_xlabel('Time (s)', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Configure which two MUs to use for control ---
MU1_IDX = 6  # <-- change to your chosen MU index
MU2_IDX = 1# <-- change to your chosen MU index

FIRING_RATE_THRESHOLD = 5  # Hz
WINDOW_SEC = 0.8         # sliding window duration in seconds

# --- Compute instantaneous firing rate with a sliding window ---
def firing_rate_sliding_window(spike_times_samples, n_total_samples, fsamp, window_sec):
    """
    Compute instantaneous firing rate at each sample using a sliding window.

    Returns an array of shape (n_total_samples,) with firing rate in Hz.
    """
    window_samples = int(window_sec * fsamp)
    half_win = window_samples // 2

    # Build a binary spike train
    spike_train = np.zeros(n_total_samples)
    valid = spike_times_samples[(spike_times_samples >= 0) & (spike_times_samples < n_total_samples)]
    spike_train[valid.astype(int)] = 1.0

    # Cumulative sum for fast window counting
    cumsum = np.cumsum(spike_train)
    cumsum = np.insert(cumsum, 0, 0)  # prepend 0 for indexing

    fr = np.zeros(n_total_samples)
    for t in range(n_total_samples):
        t_start = max(0, t - half_win)
        t_end = min(n_total_samples, t + half_win)
        actual_window = (t_end - t_start) / fsamp
        n_spikes_in_win = cumsum[t_end] - cumsum[t_start]
        fr[t] = n_spikes_in_win / actual_window if actual_window > 0 else 0.0

    return fr

n_samples = sources[0].shape[0]

fr_mu1 = firing_rate_sliding_window(recalibrated_spikes[MU1_IDX], n_samples, fsamp, WINDOW_SEC)
fr_mu2 = firing_rate_sliding_window(recalibrated_spikes[MU2_IDX], n_samples, fsamp, WINDOW_SEC)

# --- Classify each sample ---
# 0 = rest, 1 = movement 1 (both firing), 2 = movement 2 (only MU2 firing)
mu1_active = fr_mu1 >= FIRING_RATE_THRESHOLD
mu2_active = fr_mu2 >= FIRING_RATE_THRESHOLD

movement_label = np.zeros(n_samples, dtype=int)  # default rest
movement_label[mu1_active & mu2_active] = 1       # both firing -> movement 1
movement_label[mu1_active & ~mu2_active] = 1      # only MU1 firing -> movement 1
movement_label[~mu1_active & mu2_active] = 2      # only MU2 -> movement 2

# --- Plot raster with colored movement regions ---
time_s = np.arange(n_samples) / fsamp
emg_ch = 0  # EMG channel to display

fig, axes = plt.subplots(
    4, 1,
    figsize=(16, 10),
    sharex=True,
    gridspec_kw={'height_ratios': [2.5, 1, 1, 1.5]}
)

# --- Panel 0: EMG with colored vspans ---
ax_emg = axes[0]
ax_emg.plot(time_s, concatenated_emg[emg_ch, :], 'k-', linewidth=0.4, alpha=0.8)
ax_emg.set_ylabel(f'EMG Ch {emg_ch}', fontsize=10)
ax_emg.set_title(
    f'Movement Discrimination — MU {MU1_IDX} & MU {MU2_IDX} '
    f'(threshold={FIRING_RATE_THRESHOLD} Hz, window={WINDOW_SEC}s)',
    fontsize=12, fontweight='bold'
)
ax_emg.grid(True, alpha=0.3)

# Draw colored spans for each movement region — process each label separately
from matplotlib.patches import Patch
color_map = {1: ('blue', 'Movement 1 (both)'), 2: ('orange', 'Movement 2 (MU2 only)')}

for label_val, (color, name) in color_map.items():
    mask = (movement_label == label_val).astype(int)
    d = np.diff(np.concatenate(([0], mask, [0])))
    starts = np.where(d == 1)[0]
    ends = np.where(d == -1)[0]
    for s, e in zip(starts, ends):
        ax_emg.axvspan(s / fsamp, e / fsamp, color=color, alpha=0.25)

ax_emg.legend(handles=[Patch(color='blue', alpha=0.25, label='Movement 1 (both)'),
                        Patch(color='orange', alpha=0.25, label='Movement 2 (MU2 only)')],
              loc='upper right', fontsize=9)

# --- Panel 1: MU1 raster ---
ax1 = axes[1]
spk1 = recalibrated_spikes[MU1_IDX]
if len(spk1) > 0:
    ax1.scatter(spk1 / fsamp, np.zeros_like(spk1), c='tab:blue', marker='|', s=100, linewidths=2)
ax1.set_ylabel(f'MU {MU1_IDX}', fontsize=10, color='tab:blue')
ax1.set_ylim(-0.5, 0.5)
ax1.set_yticks([])
ax1.grid(True, alpha=0.3, axis='x')

# --- Panel 2: MU2 raster ---
ax2 = axes[2]
spk2 = recalibrated_spikes[MU2_IDX]
if len(spk2) > 0:
    ax2.scatter(spk2 / fsamp, np.zeros_like(spk2), c='tab:orange', marker='|', s=100, linewidths=2)
ax2.set_ylabel(f'MU {MU2_IDX}', fontsize=10, color='tab:orange')
ax2.set_ylim(-0.5, 0.5)
ax2.set_yticks([])
ax2.grid(True, alpha=0.3, axis='x')

# --- Panel 3: Firing rates ---
ax_fr = axes[3]
ax_fr.plot(time_s, fr_mu1, color='tab:blue', linewidth=1, label=f'MU {MU1_IDX} FR')
ax_fr.plot(time_s, fr_mu2, color='tab:orange', linewidth=1, label=f'MU {MU2_IDX} FR')
ax_fr.axhline(FIRING_RATE_THRESHOLD, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'Threshold ({FIRING_RATE_THRESHOLD} Hz)')
ax_fr.set_ylabel('Firing Rate (Hz)', fontsize=10)
ax_fr.set_xlabel('Time (s)', fontsize=10)
ax_fr.legend(loc='upper right', fontsize=9)
ax_fr.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Print summary ---
total_time = n_samples / fsamp
rest_time = np.sum(movement_label == 0) / fsamp
mov1_time = np.sum(movement_label == 1) / fsamp
mov2_time = np.sum(movement_label == 2) / fsamp

print(f"Total duration: {total_time:.1f} s")
print(f"  Rest:         {rest_time:.1f} s ({100*rest_time/total_time:.1f}%)")
print(f"  Movement 1:   {mov1_time:.1f} s ({100*mov1_time/total_time:.1f}%) — both MU{MU1_IDX} & MU{MU2_IDX} > {FIRING_RATE_THRESHOLD} Hz")
print(f"  Movement 2:   {mov2_time:.1f} s ({100*mov2_time/total_time:.1f}%) — only MU{MU2_IDX} > {FIRING_RATE_THRESHOLD} Hz")


In [ ]:
filename = r"C:\Users\velar\Documents\GitHub\PatientGUI\datafile1_filtered.npz"

np.savez( filename, data= concatenated_emg)

In [ ]:
import matplotlib.pyplot as plt


def calculate_spike_triggered_average(emg_signal, spike_times, fs=2048, window_ms=40):
    """
    Calculate spike-triggered average waveform and per-channel SNR.

    Args:
        emg_signal: EMG signal array of shape (n_channels, n_timepoints)
        spike_times: Array of spike times in samples
        fs: Sampling frequency in Hz
        window_ms: Window size around spike in milliseconds (total window, centered on spike)

    Returns:
        sta: Spike-triggered average waveform (n_channels, window_samples)
        time_axis: Time axis in milliseconds relative to spike
        n_spikes_used: Number of spikes used (excludes those too close to edges)
        snr_db: Per-channel SNR in dB, computed as 10*log10(||STA||^2 / mean(||wf_i - STA||^2))
    """
    # Calculate window in samples
    window_samples = int((window_ms / 1000) * fs)
    half_window = window_samples // 2

    # Get dimensions
    n_channels, n_timepoints = emg_signal.shape

    # Initialize array to collect waveforms
    waveforms = []

    # Extract waveforms around each spike
    for spike in spike_times:
        spike_idx = int(spike)
        start_idx = spike_idx - half_window
        end_idx = spike_idx + half_window

        # Check if window is within signal bounds
        if start_idx >= 0 and end_idx < n_timepoints:
            waveform = emg_signal[:, start_idx:end_idx]
            waveforms.append(waveform)

    # Calculate average and SNR
    if len(waveforms) > 0:
        waveforms = np.array(waveforms)  # shape: (n_spikes, n_channels, window_samples)
        sta = np.mean(waveforms, axis=0)  # shape: (n_channels, window_samples)
        actual_samples = sta.shape[-1]

        # Residual noise SNR per channel
        # signal power = mean of STA squared (per channel)
        signal_power = np.mean(sta ** 2, axis=-1)  # (n_channels,)
        # noise power = mean residual variance across spikes (per channel)
        residuals = waveforms - sta[np.newaxis, :, :]  # (n_spikes, n_channels, window_samples)
        noise_power = np.mean(np.mean(residuals ** 2, axis=-1), axis=0)  # (n_channels,)
        # SNR in dB, guard against division by zero
        with np.errstate(divide='ignore', invalid='ignore'):
            snr_db = 10 * np.log10(signal_power / noise_power)
            snr_db = np.where(np.isfinite(snr_db), snr_db, 0.0)
    else:
        sta = np.zeros((n_channels, window_samples))
        actual_samples = window_samples
        snr_db = np.zeros(n_channels)

    # Create time axis in milliseconds, centered at 0
    time_axis = np.linspace(0, window_ms, actual_samples)

    return sta, time_axis, len(waveforms), snr_db


def plot_sta_grid(sta, time_axis, title="Spike-Triggered Average", grid_shape=None, snr_db=None):
    """
    Plot STA as a grid of electrode positions.

    Args:
        sta: STA waveform of shape (n_channels, window_samples)
        time_axis: Time axis in milliseconds
        title: Plot title
        grid_shape: Tuple (n_rows, n_cols) for arranging channels in a grid.
                   If None, arranges channels in a square-ish grid.
        snr_db: Optional per-channel SNR in dB. If provided, shown in subplot titles.
    """
    n_channels, window_samples = sta.shape

    # Determine grid layout
    if grid_shape is not None:
        n_rows, n_cols = grid_shape
        if n_rows * n_cols < n_channels:
            raise ValueError(f"Grid shape {grid_shape} too small for {n_channels} channels")
    else:
        # Arrange in roughly square grid
        n_cols = int(np.ceil(np.sqrt(n_channels)))
        n_rows = int(np.ceil(n_channels / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*1.5, n_rows*1.5))

    # Handle single row or column case
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)

    # Find global min/max for consistent y-axis
    vmin, vmax = sta.min(), sta.max()

    channel_idx = 0
    for row in range(n_rows):
        for col in range(n_cols):
            ax = axes[row, col]

            if channel_idx < n_channels:
                # Plot channel data
                ax.plot(time_axis, sta[channel_idx, :], 'b-', linewidth=0.8)
                ax.axvline(time_axis[len(time_axis)//2], color='r', linestyle='--', alpha=0.3, linewidth=0.5)
                ax.set_ylim(vmin, vmax)

                # Show SNR in title if available
                if snr_db is not None:
                    ax.set_title(f'Ch{channel_idx} ({snr_db[channel_idx]:.1f}dB)', fontsize=7, pad=2)
                else:
                    ax.set_title(f'Ch{channel_idx}', fontsize=8, pad=2)
                channel_idx += 1
            else:
                # Hide unused subplots
                ax.axis('off')

            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle(title, fontsize=14, y=0.995)
    plt.tight_layout()

    return fig

In [ ]:
fs = 2000
window_ms = 50
stas_dict = {}


spike_times = recalibrated_spikes
for n_idx in [MU1_IDX, MU2_IDX]:

    sta, time_axis, n_spikes_used, snr_db = calculate_spike_triggered_average(
        concatenated_emg, spike_times[n_idx], fs=fs, window_ms=window_ms
    )

    time_axis = time_axis

    neuron_idx = n_idx

    stas_dict[neuron_idx] = {
        'sta': sta,
        'time_axis': time_axis,
        'n_spikes': len(spike_times[n_idx]),
        'n_spikes_used': n_spikes_used,
        'snr_db': snr_db,
    }

    print(f"Neuron {n_idx}: {n_spikes_used} spikes, mean SNR = {np.mean(snr_db):.1f} dB")
    print(f"  Per-channel SNR (dB): {np.round(snr_db, 1)}")
    print(sta.shape)


    fig = plot_sta_grid(stas_dict[neuron_idx]['sta'], stas_dict[neuron_idx]['time_axis'],
                        title="MUAPs", snr_db=stas_dict[neuron_idx]['snr_db'])

    plt.show()
